# English → Indic NMT with OpenNMT-py (ILCI subset of BPCC), CPU version (fixed)
Steps: (1) load ILCI subset of `ai4bharat/BPCC`, (2) length-binned train/valid/test splits (4 bins), (3) train a Transformer with OpenNMT-py, (4) translate the test set, (5) sacreBLEU overall and bin-wise, (6) analysis.

## 0. Environment check

In [1]:
import sys, subprocess, platform, os, glob, re
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # avoids a common OpenMP crash on macOS
print(sys.executable)
print("Machine:", platform.machine(), "| Python:", platform.python_version())

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

pip_install("OpenNMT-py", "sentencepiece", "sacrebleu", "datasets", "huggingface_hub", "pandas")

r = subprocess.run([sys.executable, "-c", "import onmt, pyonmttok; print('ok')"], capture_output=True, text=True)
print("OpenNMT-py import check:", "OK" if r.returncode == 0 else "FAILED\n" + r.stderr[-1500:])
assert r.returncode == 0

/Users/adityakumar/Desktop/miniconda3/envs/DL/bin/python
Machine: arm64 | Python: 3.11.14
OpenNMT-py import check: OK


## 1. Configuration

In [2]:
TARGET_LANG = "hin_Deva"     # change to your native language
SOURCE_LANG = "eng_Latn"
TEST_FRAC   = 0.10
SEED        = 42
SPM_VOCAB   = 8000
TRAIN_STEPS = 5000
MAX_TRAIN_PER_BIN = 5000     # None = use all
MAX_TEST_PER_BIN  = 500      # None = use all
VALID_PER_BIN     = 100      # validation sentences per bin (taken from non-test data)
WORKDIR     = "mt_work"

os.makedirs(WORKDIR, exist_ok=True)

## 2. Load the ILCI subset

In [3]:
from huggingface_hub import login, list_repo_files, hf_hub_download
import pandas as pd
login()

files = list_repo_files("ai4bharat/BPCC", repo_type="dataset")
cands = [f for f in files if "ilci" in f.lower() and TARGET_LANG in f]
print("Matching files:", cands)
assert cands, "No ILCI file found for this language."

dfs = []
for f in cands:
    p = hf_hub_download("ai4bharat/BPCC", f, repo_type="dataset")
    if f.endswith(".parquet"):   d = pd.read_parquet(p)
    elif f.endswith(".csv"):     d = pd.read_csv(p)
    elif f.endswith(".tsv"):     d = pd.read_csv(p, sep="\t")
    else:                        d = pd.read_json(p, lines=True)
    dfs.append(d)
df = pd.concat(dfs, ignore_index=True)
print(df.columns.tolist(), len(df))

if "src_lang" in df.columns:
    fwd = df[(df.src_lang == SOURCE_LANG) & (df.tgt_lang == TARGET_LANG)]
    rev = df[(df.src_lang == TARGET_LANG) & (df.tgt_lang == SOURCE_LANG)].rename(columns={"src": "tgt", "tgt": "src"})
    df = pd.concat([fwd, rev], ignore_index=True)
df = df[["src", "tgt"]].dropna()
df["src"] = df["src"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
df["tgt"] = df["tgt"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
df = df[(df.src != "") & (df.tgt != "")].drop_duplicates().reset_index(drop=True)
print("Sentence pairs:", len(df))

Matching files: ['ilci/hin_Deva.tsv']
['src_lang', 'tgt_lang', 'src', 'tgt'] 165649
Sentence pairs: 165649


## 3. Length bins and train / valid / test splits
Length = whitespace words in the English sentence. bin1: ≤10, bin2: 11–20, bin3: 21–29, bin4: ≥30. Each bin is split separately. A small **validation** set (needed by OpenNMT-py) is carved from the non-test data, so the test set stays untouched.

In [4]:
def get_bin(n):
    if n <= 10: return "bin1"
    if n <= 20: return "bin2"
    if n < 30:  return "bin3"
    return "bin4"

df["len"] = df["src"].str.split().str.len()
df["bin"] = df["len"].apply(get_bin)

train_parts, valid_parts, test_parts = [], [], []
for b, g in df.groupby("bin"):
    g = g.sample(frac=1.0, random_state=SEED)
    n_test = max(1, int(round(len(g) * TEST_FRAC)))
    te, rest = g.iloc[:n_test], g.iloc[n_test:]
    va, tr = rest.iloc[:VALID_PER_BIN], rest.iloc[VALID_PER_BIN:]
    if MAX_TEST_PER_BIN is not None:  te = te.iloc[:MAX_TEST_PER_BIN]
    if MAX_TRAIN_PER_BIN is not None: tr = tr.iloc[:MAX_TRAIN_PER_BIN]
    test_parts.append(te); valid_parts.append(va); train_parts.append(tr)

train_df = pd.concat(train_parts).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
valid_df = pd.concat(valid_parts).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
test_df  = pd.concat(test_parts).reset_index(drop=True)

print(pd.DataFrame({"train": train_df["bin"].value_counts().sort_index(),
                    "valid": valid_df["bin"].value_counts().sort_index(),
                    "test":  test_df["bin"].value_counts().sort_index()}))

      train  valid  test
bin                     
bin1   5000    100   500
bin2   5000    100   500
bin3   5000    100   500
bin4   5000    100   500


## 4. SentencePiece and data files

In [5]:
import sentencepiece as spm

def write_lines(path, lines):
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

for name, d in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    write_lines(f"{WORKDIR}/{name}.src", d["src"])
    write_lines(f"{WORKDIR}/{name}.tgt", d["tgt"])

for side in ["src", "tgt"]:
    spm.SentencePieceTrainer.train(
        input=f"{WORKDIR}/train.{side}", model_prefix=f"{WORKDIR}/spm_{side}",
        vocab_size=SPM_VOCAB, character_coverage=1.0, model_type="unigram",
        pad_id=-1, bos_id=-1, eos_id=-1, unk_id=0, minloglevel=2)

sp_src = spm.SentencePieceProcessor(model_file=f"{WORKDIR}/spm_src.model")
sp_tgt = spm.SentencePieceProcessor(model_file=f"{WORKDIR}/spm_tgt.model")

def encode_file(sp, inp, out):
    with open(inp, encoding="utf-8") as f:
        lines = [l.rstrip("\n") for l in f]
    write_lines(out, [" ".join(sp.encode(l, out_type=str)) for l in lines])

encode_file(sp_src, f"{WORKDIR}/train.src", f"{WORKDIR}/train.sp.src")
encode_file(sp_tgt, f"{WORKDIR}/train.tgt", f"{WORKDIR}/train.sp.tgt")
encode_file(sp_src, f"{WORKDIR}/valid.src", f"{WORKDIR}/valid.sp.src")
encode_file(sp_tgt, f"{WORKDIR}/valid.tgt", f"{WORKDIR}/valid.sp.tgt")
encode_file(sp_src, f"{WORKDIR}/test.src",  f"{WORKDIR}/test.sp.src")
print(sorted(os.listdir(WORKDIR)))

['config.yaml', 'model_step_4000.pt', 'model_step_5000.pt', 'sample', 'spm_src.model', 'spm_src.vocab', 'spm_tgt.model', 'spm_tgt.vocab', 'test.hyp', 'test.sp.hyp', 'test.sp.src', 'test.src', 'test.tgt', 'train.sp.src', 'train.sp.tgt', 'train.src', 'train.tgt', 'valid.sp.src', 'valid.sp.tgt', 'valid.src', 'valid.tgt', 'vocab.src', 'vocab.tgt']


## 5. OpenNMT-py config, vocabulary, training (CPU)

In [6]:
config = f"""
save_data: {WORKDIR}/onmt_run
src_vocab: {WORKDIR}/vocab.src
tgt_vocab: {WORKDIR}/vocab.tgt
overwrite: true
skip_empty_level: silent

data:
    corpus_1:
        path_src: {WORKDIR}/train.sp.src
        path_tgt: {WORKDIR}/train.sp.tgt
        weight: 1
    valid:
        path_src: {WORKDIR}/valid.sp.src
        path_tgt: {WORKDIR}/valid.sp.tgt

save_model: {WORKDIR}/model
save_checkpoint_steps: 1000
keep_checkpoint: 2
train_steps: {TRAIN_STEPS}
valid_steps: 1000
report_every: 100

# CPU only
world_size: 1
gpu_ranks: []

batch_type: tokens
batch_size: 2048
valid_batch_size: 16
accum_count: [1]
accum_steps: [0]
queue_size: 1000
bucket_size: 10000
num_workers: 0

optim: adam
adam_beta1: 0.9
adam_beta2: 0.998
decay_method: noam
learning_rate: 2.0
warmup_steps: 1000
max_grad_norm: 0.0
label_smoothing: 0.1
param_init: 0.0
param_init_glorot: true
normalization: tokens

encoder_type: transformer
decoder_type: transformer
position_encoding: true
enc_layers: 3
dec_layers: 3
heads: 4
hidden_size: 256
word_vec_size: 256
transformer_ff: 1024
dropout_steps: [0]
dropout: [0.1]
attention_dropout: [0.1]
share_vocab: false
"""
with open(f"{WORKDIR}/config.yaml", "w", encoding="utf-8") as f:
    f.write(config)

def run_onmt(module, *args):
    """Run an OpenNMT-py command with this notebook's Python and stream its output."""
    cmd = [sys.executable, "-m", f"onmt.bin.{module}", *args]
    print(" ".join(cmd))
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        print(line, end="")
        tail.append(line); tail = tail[-30:]
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f"onmt.bin.{module} failed (exit {p.returncode}). Last output:\n" + "".join(tail))

# build vocab on the full training data (-n_sample -1 is only for this command)
run_onmt("build_vocab", "-config", f"{WORKDIR}/config.yaml", "-n_sample", "-1")

/Users/adityakumar/Desktop/miniconda3/envs/DL/bin/python -m onmt.bin.build_vocab -config mt_work/config.yaml -n_sample -1
[2026-10-23 17:01:31,197 INFO] Counter vocab from -1 samples.
[2026-10-23 17:01:31,197 INFO] n_sample=-1: Build vocab on full datasets.
[2026-10-23 17:01:32,511 INFO] Counters src: 7998
[2026-10-23 17:01:32,511 INFO] Counters tgt: 7997
[2026-10-23 17:01:32,512 WARNING] path mt_work/vocab.src exists, may overwrite...
[2026-10-23 17:01:32,515 WARNING] path mt_work/vocab.tgt exists, may overwrite...


In [7]:
# actual training (no -n_sample here, otherwise it only previews data and does not train)
run_onmt("train", "-config", f"{WORKDIR}/config.yaml")

/Users/adityakumar/Desktop/miniconda3/envs/DL/bin/python -m onmt.bin.train -config mt_work/config.yaml
[2026-10-23 17:01:33,828 INFO] Missing transforms field for corpus_1 data, set to default: [].
[2026-10-23 17:01:33,828 INFO] Missing transforms field for valid data, set to default: [].
[2026-10-23 17:01:33,828 INFO] Parsed 2 corpora from -data.
[2026-10-23 17:01:33,828 INFO] Get special vocabs from Transforms: {'src': [], 'tgt': []}.
[2026-10-23 17:01:33,837 INFO] The first 10 tokens of the vocabs are:['<unk>', '<blank>', '<s>', '</s>', '▁the', '▁of', 's', ',', '▁and', '.']
[2026-10-23 17:01:33,837 INFO] The decoder start token is: <s>
[2026-10-23 17:01:33,837 INFO] Building model...
[2026-10-23 17:01:34,075 INFO] Switching model to float32 for amp/apex_amp
[2026-10-23 17:01:34,075 INFO] Non quantized layer compute is fp32
[2026-10-23 17:01:34,076 INFO] NMTModel(
  (encoder): TransformerEncoder(
    (embeddings): Embeddings(
      (make_embedding): Sequential(
        (emb_luts): El

## 6. Translate the test set (CPU)

In [8]:
def step_of(p):
    m = re.search(r"model_step_(\d+)\.pt$", p)
    return int(m.group(1)) if m else -1

ckpts = sorted(glob.glob(f"{WORKDIR}/model_step_*.pt"), key=step_of)
assert ckpts, "No checkpoint found; training did not finish. Re-run the training cell and check for errors."
best_ckpt = ckpts[-1]
print("Using checkpoint:", best_ckpt)

run_onmt("translate",
         "-model", best_ckpt,
         "-src", f"{WORKDIR}/test.sp.src",
         "-output", f"{WORKDIR}/test.sp.hyp",
         "-beam_size", "5",
         "-batch_size", "32",
         "-batch_type", "sents",
         "-max_length", "200",
         "-verbose")

with open(f"{WORKDIR}/test.sp.hyp", encoding="utf-8") as f:
    hyp_sp = [l.rstrip("\n") for l in f]
hyps = [sp_tgt.decode(l.split()) for l in hyp_sp]
assert len(hyps) == len(test_df), (len(hyps), len(test_df))
test_df["hyp"] = hyps
write_lines(f"{WORKDIR}/test.hyp", hyps)
test_df[["src", "tgt", "hyp"]].head()

Using checkpoint: mt_work/model_step_5000.pt
/Users/adityakumar/Desktop/miniconda3/envs/DL/bin/python -m onmt.bin.translate -model mt_work/model_step_5000.pt -src mt_work/test.sp.src -output mt_work/test.sp.hyp -beam_size 5 -batch_size 32 -batch_type sents -max_length 200 -verbose
[2026-10-23 17:36:09,849 INFO] Loading checkpoint from mt_work/model_step_5000.pt
[2026-10-23 17:36:10,083 INFO] Loading data into the model
[2026-10-23 17:37:06,261 INFO] 
SENT 1: ['▁Which', '▁has', '▁clinical', '▁fruit', '▁smell', '▁and', '▁sharp', '▁taste', '.']
PRED 1: ▁एक ▁बार - बार ी ▁रोग ▁है ।
PRED SCORE: -1.7644

[2026-10-23 17:37:06,261 INFO] 
SENT 2: ['▁Their', '▁commission', '▁is', '▁fixed', '▁everywhere', '▁.']
PRED 2: ▁इसके ▁अलावा ▁एन् थ ि ब ्यू ट िज्म ▁रोग ▁है ▁।
PRED SCORE: -1.8290

[2026-10-23 17:37:06,262 INFO] 
SENT 3: ['▁He', 'ar', 'ing', '▁loss']
PRED 3: ▁यह ▁माना ▁जाता ▁है ।
PRED SCORE: -1.0670

[2026-10-23 17:37:06,262 INFO] 
SENT 4: ['▁Incubate', '▁the', '▁tube', '▁for', '▁15', '-', '20

,src,tgt,hyp
0,Which has clinical fruit smell and sharp taste.,जिसमें लाक्षणिक फल गंध और तीखा स्वाद होता है।,एक बार-बारी रोग है।
1,Their commission is fixed everywhere .,हर जगह इनका कमीशन बँधा रहता है ।,इसके अलावा एन्थिब्यूटिज्म रोग है ।
2,Hearing loss,बधिरता,यह माना जाता है।
3,Incubate the tube for 15-20 minutes at minus 2...,माइनस 20 सी पर 15-20 मिनट के लिए ट्यूब को इंक्...,- एसोसाइटिक स्ट्रेप्टोकोकी के लिए एंटीमाइक्रोब...
4,"So, how should you think about functions infor...","तो, आपको अनौपचारिक रूप से फंक्शन्स के बारे में...","तो, आप जानते हैं।"


## 7. BLEU evaluation (sacreBLEU): overall and bin-wise

In [9]:
from sacrebleu.metrics import BLEU, CHRF

bleu, chrf = BLEU(), CHRF()
overall = bleu.corpus_score(test_df["hyp"].tolist(), [test_df["tgt"].tolist()])
print("Overall BLEU:", overall)
print("Overall chrF:", chrf.corpus_score(test_df["hyp"].tolist(), [test_df["tgt"].tolist()]))

rows = []
for b in ["bin1", "bin2", "bin3", "bin4"]:
    g = test_df[test_df["bin"] == b]
    s = bleu.corpus_score(g["hyp"].tolist(), [g["tgt"].tolist()])
    c = chrf.corpus_score(g["hyp"].tolist(), [g["tgt"].tolist()])
    rows.append({"bin": b, "test sentences": len(g), "avg src length": round(g["len"].mean(), 1),
                 "BLEU": round(s.score, 2), "chrF": round(c.score, 2)})
bin_results = pd.DataFrame(rows)
bin_results

Overall BLEU: BLEU = 0.70 13.8/2.4/0.4/0.1 (BP = 0.623 ratio = 0.679 hyp_len = 32711 ref_len = 48169)
Overall chrF: chrF2 = 7.88


,bin,test sentences,avg src length,BLEU,chrF
0,bin1,500,7.4,1.36,9.24
1,bin2,500,15.2,0.78,8.84
2,bin3,500,24.3,0.41,8.09
3,bin4,500,37.0,0.67,6.99


## 8. Bin-wise result analysis

In [10]:
best  = bin_results.loc[bin_results["BLEU"].idxmax()]
worst = bin_results.loc[bin_results["BLEU"].idxmin()]
print(f"Overall BLEU: {overall.score:.2f}")
print(f"Highest BLEU: {best['bin']} ({best['BLEU']}), lowest BLEU: {worst['bin']} ({worst['BLEU']}).")
print("BLEU by bin (short -> long):", bin_results["BLEU"].tolist())
trend = bin_results["BLEU"].diff().dropna()
if (trend < 0).all():   print("BLEU decreases steadily as sentence length increases.")
elif (trend > 0).all(): print("BLEU increases steadily as sentence length increases.")
else:                   print("BLEU does not change monotonically with sentence length.")

Overall BLEU: 0.70
Highest BLEU: bin1 (1.36), lowest BLEU: bin3 (0.41).
BLEU by bin (short -> long): [1.36, 0.78, 0.41, 0.67]
BLEU does not change monotonically with sentence length.


**Analysis (fill in after running):** Compare BLEU across bin1–bin4. Points to discuss: how BLEU changes as source length grows (long sentences usually score lower because errors accumulate and the model sees fewer long examples), whether very short sentences score unusually high or low (few n-grams make BLEU unstable), the number of test sentences per bin (small bins give noisy scores), and the amount of training data per bin. This CPU run uses a small model, a capped training set and few steps, so absolute BLEU will be low.